In [0]:
import json

In [0]:
run_date = dbutils.widgets.get("run_date")
table_config_path = dbutils.widgets.get("table_config_path")

with open(table_config_path, "r") as file:
    table_config = json.load(file)

db, table_names = table_config["DATABASE"], table_config["TABLE_NAMES"]
print(f"Input parameter run_date:{run_date}")

In [0]:
def combine_lpo_mmc_results(spark, run_date, db, table_names):
    df = spark.sql("""
                    SELECT vin, product_id, product_type, 'MMC' as product_category, score, dt 
                    FROM {db}.{mmc_final_results}
                    WHERE dt = '{run_date}'
                    UNION ALL
                    SELECT vin, product_id, product_type, 'LPO' as product_category, score, dt
                    FROM {db}.{lpo_final_results}
                    WHERE dt = '{run_date}'
                    ORDER BY vin, product_category, product_type, product_id
                    """.format(run_date = run_date,
                               db = db,
                               mmc_final_results = table_names['mmc_final_results'],
                               lpo_final_results = table_names['lpo_final_results']))
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(db+'.'+table_names["combined_results"])

In [0]:
combine_lpo_mmc_results(spark, run_date, db, table_names)